# Do Wind-Directed Graphs Help? Delhi PM2.5 forecasting experiments

**Paper:** *Do Wind-Directed Graphs Add Information Beyond Station History? A Transport-Conditioned Evaluation on Delhi's Air-Quality Network*
**Authors:** Raushan Kumar, Priya Sahoo, Grandhi Saketh (SRM University-AP)
**Code:** https://github.com/raushan95a/wind-directed-graphs-delhi-aq

This notebook reproduces every experiment in the paper, from downloading the data to the final statistics. Nothing needs to be uploaded and no API keys are required.

**Research question.** Many recent forecasting models connect monitoring stations with edges that follow the wind. Does orienting edges with the wind actually add information beyond each station's own history and beyond an ordinary neighbour average? We answer this by keeping the learning algorithm fixed and changing only the graph.

---
### How to run
1. **Runtime > Change runtime type > T4 GPU > Save.**
2. **Runtime > Run all**, and approve the Google Drive prompt (outputs are kept in Drive so a disconnect loses nothing).
3. If Colab disconnects, choose **Run all** again: finished steps are detected and skipped.
4. At the end `final_results.zip` downloads automatically; a copy stays in `MyDrive/delhi_windgraph/`.

### Contents
| Step | Purpose | Approx. time (T4, estimate) |
|---|---|---|
| 0 | Settings | - |
| 1 | Install requirements, check GPU, choose working folder | 1 min |
| 2 | Record hardware and software | seconds |
| 3 | Write the pipeline scripts and station coordinates to files | seconds |
| 4 | Download Delhi CPCB data, 2019-2025 | 5-10 min |
| 5 | Build hourly panel, quality control, ERA5 meteorology | 3-5 min |
| 6 | Model-free transport test | about 5 min |
| 7 | Main experiment: 7 graphs x 5 horizons, 3 test years (XGBoost) | 1.5-3 h |
| 8 | Robustness check with a graph neural network | 45-75 min |
| 9 | Training curves and resource use | 30-45 min |
| 10 | Package and download results | 1 min |
| 11 | Summary of outputs and troubleshooting | - |

### Faculty requirements covered
| Requirement | Where |
|---|---|
| Experimental setup: dataset, splits, hardware/software, hyperparameters | Steps 2, 4, 5, 7 |
| Algorithm comparison vs. baselines (MAE, RMSE, R2, significance tests) | Step 7 |
| Ablation study (edge direction, static vs. hourly wind, ventilation gate) | Step 7 |
| Training dynamics (loss curves) | Step 9 |
| Resource consumption (time, GPU memory, inference latency) | Step 9 |
| Code integrity | this notebook + GitHub repository |

## 0. Settings
Change these only if you need a shorter run. Each test year trains on all years up to two years earlier, tunes on the year before, and tests on that year, so no future data are ever used for training.

In [ ]:
# ---- experiment settings -------------------------------------------------------------
YEARS         = [2025, 2024, 2023]  # rolling test years: train <= Y-2, validate Y-1, test Y
END_YEAR      = 2025                # last year of data to download (2026 is not yet in the archive)
GNN_YEAR      = 2025                # test year for the graph-neural-network robustness check
GNN_SEEDS     = "0,1,2"             # random seeds for the GNN; "0" roughly halves its run time
RESOURCE_YEAR = 2025                # test year used for training-curve / resource measurements
USE_DRIVE     = True                # save outputs to Google Drive (safe against disconnects)

## 1. Requirements and working folder
Installs the few packages Colab does not already have, confirms that a GPU is available, and creates the working folder (`src/` for code, `data/` for downloads, `out/` for results).

In [ ]:
# ---- install packages (most are preinstalled on Colab) -------------------------------
!pip -q install fsspec aiohttp requests pyarrow xgboost scipy matplotlib pandas scikit-learn
import os, sys, json
import torch, xgboost

# ---- confirm the GPU ----------------------------------------------------------------
print('python', sys.version.split()[0], '| torch', torch.__version__, '| xgboost', xgboost.__version__)
print('GPU available:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-> Runtime > Change runtime type > T4 GPU')

# ---- working folder: Google Drive if possible, otherwise the local Colab disk ---------
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK = '/content/drive/MyDrive/delhi_windgraph'
    except Exception as e:
        print('Drive not mounted (', e, ') -> using local disk')
        WORK = '/content/delhi_windgraph'
else:
    WORK = '/content/delhi_windgraph'
for d in ['src', 'data', 'out']:
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.chdir(WORK)
print('working in', os.getcwd())

## 2. Hardware and software record
Saves the GPU model, memory and library versions to `out/env_info.json`. These values go into the paper's *Experimental Setup* subsection.

In [ ]:
import platform, pandas, numpy, scipy, sklearn

info = {
    "python": sys.version.split()[0], "torch": torch.__version__, "xgboost": xgboost.__version__,
    "pandas": pandas.__version__, "numpy": numpy.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if torch.cuda.is_available() else None,
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1),
    "platform": platform.platform(),
}
json.dump(info, open('out/env_info.json', 'w'), indent=1)
print(json.dumps(info, indent=1))

## 3. Pipeline scripts and inputs
Each cell below writes one file. Keeping the code in separate scripts (identical to the GitHub repository) makes every step easy to read and re-run on its own.

| File | Role |
|---|---|
| `data/station_coords.csv` | Station latitude/longitude (XKDR catalogue) |
| `src/download_data.py` | Download the Delhi rows of the CPCB archive |
| `src/prepare_data.py` | Hourly panel, quality control, station selection, ERA5 |
| `src/lag_test.py` | Model-free upwind/downwind test |
| `src/graph_experiment.py` | Main graph comparison with XGBoost |
| `src/evaluate_stats.py` | Metrics, significance tests, regime breakdowns |
| `src/gnn_experiment.py` | Graph neural network robustness check |
| `src/resources.py` | Training curves and resource measurements |

**Station coordinates.** Latitude and longitude of each CPCB station, used to compute distances and wind bearings between stations.

In [ ]:
%%writefile data/station_coords.csv
station_id,state,city,station_name,lat,lon
site_103,Delhi,Delhi,"CRRI Mathura Road, Delhi - IMD",28.5512005,77.2735737
site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",28.7256504,77.2011573
site_105,Delhi,Delhi,"North Campus, DU, Delhi - IMD",28.6573814,77.1585447
site_106,Delhi,Delhi,"IGI Airport (T3), Delhi - IMD",28.5627763,77.1180053
site_107,Delhi,Delhi,"Pusa, Delhi - IMD",28.639652,77.146275
site_108,Delhi,Delhi,"Aya Nagar, Delhi - IMD",28.4706914,77.1099364
site_109,Delhi,Delhi,"Lodhi Road, Delhi - IMD",28.5918245,77.2273074
site_111,Uttar Pradesh,Noida,"Sector - 62, Noida - IMD",28.6245479,77.3577104
site_113,Delhi,Delhi,"Shadipur, Delhi - CPCB",28.6514781,77.1473105
site_114,Delhi,Delhi,"IHBAS, Dilshad Garden, Delhi - CPCB",28.6811736,77.3025234
site_115,Delhi,Delhi,"NSIT Dwarka, Delhi - CPCB",28.60909,77.0325413
site_117,Delhi,Delhi,"ITO, Delhi - CPCB",28.628624,77.24106
site_118,Delhi,Delhi,"DTU, Delhi - CPCB",28.7500499,77.1112615
site_119,Delhi,Delhi,"Sirifort, Delhi - CPCB",28.5504249,77.2159377
site_122,Delhi,Delhi,"Mandir Marg, Delhi - DPCC",28.636429,77.201067
site_124,Delhi,Delhi,"R K Puram, Delhi - DPCC",28.563262,77.186937
site_125,Delhi,Delhi,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023
site_1420,Delhi,Delhi,"Ashok Vihar, Delhi - DPCC",28.695381,77.181665
site_1421,Delhi,Delhi,"Dr. Karni Singh Shooting Range, Delhi - DPCC",28.498571,77.26484
site_1422,Delhi,Delhi,"Dwarka-Sector 8, Delhi - DPCC ",28.5710274,77.0719006
site_1423,Delhi,Delhi,"Jahangirpuri, Delhi - DPCC",28.73282,77.170633
site_1424,Delhi,Delhi,"Jawaharlal Nehru Stadium, Delhi - DPCC",28.58028,77.233829
site_1425,Delhi,Delhi,"Major Dhyan Chand National Stadium, Delhi - DPCC",28.611281,77.237738
site_1426,Delhi,Delhi,"Narela, Delhi - DPCC",28.822836,77.101981
site_1427,Delhi,Delhi,"Najafgarh, Delhi - DPCC",28.570173,76.933762
site_1428,Delhi,Delhi,"Okhla Phase-2, Delhi - DPCC",28.530785,77.271255
site_1429,Delhi,Delhi,"Nehru Nagar, Delhi - DPCC",28.56789,77.250515
site_1430,Delhi,Delhi,"Rohini, Delhi - DPCC",28.732528,77.11992
site_1431,Delhi,Delhi,"Patparganj, Delhi - DPCC",28.623763,77.287209
site_1432,Delhi,Delhi,"Sonia Vihar, Delhi - DPCC",28.710508,77.249485
site_1434,Delhi,Delhi,"Wazirpur, Delhi - DPCC",28.699793,77.165453
site_1435,Delhi,Delhi,"Vivek Vihar, Delhi - DPCC",28.672342,77.31526
site_144,Uttar Pradesh,Ghaziabad,"Vasundhara, Ghaziabad - UPPCB",28.6603346,77.3572563
site_146,Haryana,Gurugram,"Vikas Sadan, Gurugram - HSPCB",28.4501238,77.0263051
site_153,Uttar Pradesh,Noida,"Sector - 125, Noida - UPPCB",28.5447608,77.3231257
site_1560,Delhi,Delhi,"Bawana, Delhi - DPCC",28.7762,77.051074
site_1561,Delhi,Delhi,"Mundka, Delhi - DPCC",28.684678,77.076574
site_1562,Delhi,Delhi,"Sri Aurobindo Marg, Delhi - DPCC",28.531346,77.190156
site_1563,Delhi,Delhi,"Pusa, Delhi - DPCC",28.639652,77.146275
site_263,Haryana,Faridabad,"Sector- 16A, Faridabad - HSPCB",28.376058,77.315741
site_301,Delhi,Delhi,"Anand Vihar, Delhi - DPCC",28.647622,77.315809
site_5024,Delhi,Delhi,"Alipur, Delhi - DPCC",28.815329,77.15301
site_5025,Haryana,Gurugram,"NISE Gwal Pahari, Gurugram - IMD",28.422681,77.148944
site_5081,Uttar Pradesh,Ghaziabad,"Sanjay Nagar, Ghaziabad - UPPCB",28.685382,77.453839
site_5082,Uttar Pradesh,Ghaziabad,"Indirapuram, Ghaziabad - UPPCB",28.646233,77.358075
site_5083,Uttar Pradesh,Ghaziabad,"Loni, Ghaziabad - UPPCB",28.757294,77.278792
site_5122,Uttar Pradesh,Noida,"Sector-116, Noida - UPPCB",28.56923,77.393848
site_5123,Uttar Pradesh,Noida,"Sector-1, Noida - UPPCB",28.5898,77.3101
site_5340,Haryana,Faridabad,"New Industrial Town, Faridabad - HSPCB",28.39072,77.30059
site_5341,Haryana,Faridabad,"Sector 30, Faridabad - HSPCB",28.4417,77.3217
site_5342,Haryana,Faridabad,"Sector 11, Faridabad - HSPCB",28.376058,77.315741
site_5344,Haryana,Gurugram,"Teri Gram, Gurugram - HSPCB",28.4275,77.1465
site_5345,Haryana,Gurugram,"Sector-51, Gurugram - HSPCB",28.4227,77.0667
site_5393,Delhi,Delhi,"Chandni Chowk, Delhi - IITM",28.656756,77.227234
site_5395,Delhi,Delhi,"Lodhi Road, Delhi - IITM",28.5918245,77.2273074
site_6060,Delhi,Delhi,"IIT Delhi, Delhi - IITM",,


**`download_data.py`** - reads only the Delhi rows of each yearly archive file using HTTP range requests.

In [ ]:
%%writefile src/download_data.py
"""Step 0: download Delhi rows of the CPCB archive (Vonter/india-cpcb-aqi, ODbL-1.0) for 2019-2024.

Reads only the Delhi row groups and needed columns from the remote yearly parquet files via HTTP range
requests (~75 MB total). Falls back to downloading the full yearly file if range reads fail.
Usage: python download_data.py <out_dir> 2019 2020 ... 2024
"""
import sys, os, subprocess
import fsspec, pyarrow.parquet as pq, pyarrow.compute as pc

outdir = sys.argv[1]; years = [int(y) for y in sys.argv[2:]]; os.makedirs(outdir, exist_ok=True)
WANT = ("Station ID", "State", "City", "Station Name", "Timestamp", "PM2.5", "PM10", "AT", "RH", "WS", "WD")

# Keep only the row groups whose 'State' range can contain Delhi, read just the needed
# columns, then filter to Delhi rows and drop the unit suffixes from column names.
def extract(pf):
    names = pf.schema_arrow.names
    want = [n for n in names if n.split(" (")[0] in WANT]
    si = names.index("State"); md = pf.metadata
    rgs = [i for i in range(md.num_row_groups)
           if (lambda st: st is None or not st.has_min_max or (st.min <= "Delhi" <= st.max))(md.row_group(i).column(si).statistics)]
    t = pf.read_row_groups(rgs, columns=want)
    t = t.filter(pc.equal(t["State"], "Delhi"))
    return t.rename_columns([n.split(" (")[0] for n in t.column_names])

# One remote parquet file per year. Range reads avoid downloading the full ~0.3-0.8 GB file;
# if the server refuses range requests we fall back to a full download.
for y in years:
    dst = os.path.join(outdir, f"delhi_{y}.parquet")
    if os.path.exists(dst): print(y, "already present"); continue
    url = f"https://github.com/Vonter/india-cpcb-aqi/releases/download/{y}/cpcb-air-quality-{y}.parquet"
    try:
        fs = fsspec.filesystem("http")
        with fs.open(url, "rb", block_size=4 * 2 ** 20) as f:
            t = extract(pq.ParquetFile(f))
    except Exception as e:
        print(y, "range read failed (", e, ") -> full download", flush=True)
        tmp = f"/tmp/cpcb_{y}.parquet"
        subprocess.run(["wget", "-q", "-O", tmp, url], check=True)
        t = extract(pq.ParquetFile(tmp)); os.remove(tmp)
    pq.write_table(t, dst)
    print(y, "rows", t.num_rows, "stations", len(pc.unique(t["Station ID"])), flush=True)


**`prepare_data.py`** - hourly PM2.5 panel, quality control, station selection, ERA5 meteorology.

In [ ]:
%%writefile src/prepare_data.py
"""Step 1: build the hourly Delhi PM2.5 panel and ERA5 meteorology.

Usage: python prepare_data.py <raw_parquet_dir> <coords_csv> <out_dir> [end_year=2025]

Outputs (out_dir):
  panel.npz      pm[T,N] hourly PM2.5 (NaN = missing), times (IST, naive), station ids/names, lat, lon
  met.parquet    hourly ERA5 (via Open-Meteo, IST) per ERA5 grid point + station -> grid map
  stations.csv   selected stations with completeness
Notes:
  * Archive timestamps are labelled UTC but are local IST (verified from diurnal cycle); we treat them as IST.
  * Quarter-hour values are near-duplicates, so an hour is valid if >=1 quarter-hour value is valid.
"""
import sys, os, glob, json, time
import numpy as np, pandas as pd, requests

raw_dir, coords_csv, out = sys.argv[1], sys.argv[2], sys.argv[3]
END = int(sys.argv[4]) if len(sys.argv) > 4 else 2025
os.makedirs(out, exist_ok=True)

# =====================================================================================
# 1) Hourly PM2.5 panel with quality control
#    - timestamps in the archive are labelled UTC but are local IST, so we keep the wall clock
#    - an hour is valid if at least one quarter-hour value is valid (quarter-hours are near-duplicates)
# =====================================================================================
# ---------- PM2.5 ----------
df = pd.concat([pd.read_parquet(f, columns=["Station ID", "Station Name", "Timestamp", "PM2.5", "PM10"])
                for f in sorted(glob.glob(os.path.join(raw_dir, "delhi_20*.parquet")))], ignore_index=True)
df["t"] = df["Timestamp"].dt.tz_localize(None).dt.floor("h")          # IST wall clock
pm = df["PM2.5"].where((df["PM2.5"] >= 2) & (df["PM2.5"] < 999))       # QC: detection limit, cap
H = df.assign(pm=pm).groupby(["Station ID", "t"]).agg(pm=("pm", "mean"), pm10=("PM10", "mean")).reset_index()
H.loc[(H.pm10.notna()) & (H.pm > H.pm10 * 1.05), "pm"] = np.nan       # QC: PM2.5 > PM10
names = df.groupby("Station ID")["Station Name"].first()

times = pd.date_range("2019-01-01 00:00", f"{END}-12-31 23:00", freq="h")
P = H.pivot(index="t", columns="Station ID", values="pm").reindex(times)

# flat-line QC: runs of >=6 identical hourly values -> missing (sensor stuck)
for c in P.columns:
    s = P[c]
    grp = (s != s.shift()).cumsum()
    run = s.groupby(grp).transform("size")
    P.loc[(run >= 6) & s.notna(), c] = np.nan

# 2) Station selection: complete enough in every training year and usable in later years.
# station selection: >=75% completeness in each year 2019-2022, and >=50% in every later year up to END
comp = P.notna().groupby(P.index.year).mean().T
keep = comp.index[(comp[[2019, 2020, 2021, 2022]] >= 0.75).all(axis=1) & (comp[list(range(2023, END + 1))] >= 0.50).all(axis=1)]
coords = pd.read_csv(coords_csv).set_index("station_id")
keep = [k for k in keep if k in coords.index]
P = P[keep]
lat = coords.loc[keep, "lat"].values.astype(float); lon = coords.loc[keep, "lon"].values.astype(float)
st = pd.DataFrame({"station_id": keep, "name": names.loc[keep].values, "lat": lat, "lon": lon})
st = st.join(comp.loc[keep].add_prefix("comp_").reset_index(drop=True))
st.to_csv(os.path.join(out, "stations.csv"), index=False)
print(f"stations kept: {len(keep)} of {comp.shape[0]}")
print("overall completeness of kept panel:", round(P.notna().values.mean(), 3))

# =====================================================================================
# 3) Meteorology: ERA5 at the 0.25 deg grid points nearest to the stations
#    (10 m and 100 m wind, boundary-layer height), requested directly in IST.
# =====================================================================================
# ---------- ERA5 via Open-Meteo (archive API, models=era5, IST) ----------
grid = sorted({(round(a * 4) / 4, round(b * 4) / 4) for a, b in zip(lat, lon)})
st_grid = [grid.index((round(a * 4) / 4, round(b * 4) / 4)) for a, b in zip(lat, lon)]
vars_ = "wind_speed_10m,wind_direction_10m,wind_speed_100m,wind_direction_100m,boundary_layer_height"
mets = []
for gi, (a, b) in enumerate(grid):
    for attempt in range(5):
        r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=dict(
            latitude=a, longitude=b, start_date="2019-01-01", end_date=f"{END}-12-31", hourly=vars_,
            wind_speed_unit="ms", timezone="Asia/Kolkata", models="era5"), timeout=120)
        if r.ok: break
        time.sleep(10)
    r.raise_for_status()
    h = pd.DataFrame(r.json()["hourly"]); h["time"] = pd.to_datetime(h["time"]); h["grid"] = gi
    h["grid_lat"], h["grid_lon"] = a, b
    mets.append(h)
    print("grid", gi, (a, b), "rows", len(h), "stations", st_grid.count(gi))
met = pd.concat(mets, ignore_index=True)
met.to_parquet(os.path.join(out, "met.parquet"), index=False)

# 4) Save the panel (time x station matrix) plus station metadata for all later steps.
np.savez_compressed(os.path.join(out, "panel.npz"), pm=P.values.astype(np.float32),
                    times=times.values.astype("datetime64[h]").astype(np.int64), station_ids=np.array(keep),
                    names=names.loc[keep].values.astype(str), lat=lat, lon=lon, st_grid=np.array(st_grid))
json.dump({"n_stations": len(keep), "grid": grid, "T": len(times)}, open(os.path.join(out, "prep_meta.json"), "w"), indent=1)
print("done")


**`lag_test.py`** - model-free upwind/downwind lagged-correlation test.

In [ ]:
%%writefile src/lag_test.py
"""Step 2: model-free test for a wind-transport signature between Delhi stations (hypothesis H1).

For each ordered station pair (i -> j), compare lagged correlations of hourly log-PM2.5 changes
  c(tau) = corr( dz_i(t), dz_j(t+tau) )
on hours when the city wind blows from i towards j ("downwind" set) versus from j towards i ("upwind" set).
Transport predicts c_down(tau) > c_up(tau) for small positive lags.

Usage: python lag_test.py <prep_dir> <out_dir>
"""
import sys, os, json
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

prep, out = sys.argv[1], sys.argv[2]; os.makedirs(out, exist_ok=True)
DESEASON = len(sys.argv) > 3 and sys.argv[3] == "deseason"
Z = np.load(os.path.join(prep, "panel.npz"), allow_pickle=True)
pm, lat, lon = Z["pm"], Z["lat"], Z["lon"]
times = pd.to_datetime(Z["times"].astype("datetime64[h]"))
N = pm.shape[1]
met = pd.read_parquet(os.path.join(prep, "met.parquet"))

# =====================================================================================
# 1) City-mean wind and its steadiness over the next 6 h (direction must stay constant
#    for transport between stations to be meaningful).
# =====================================================================================
# city-mean 100 m wind (vector towards which air moves), averaged over the 4 grid points
def uv(s, d):  # meteorological direction = from
    r = np.deg2rad(d); return -s * np.sin(r), -s * np.cos(r)
met["u"], met["v"] = uv(met.wind_speed_100m, met.wind_direction_100m)
W = met.groupby("time")[["u", "v"]].mean().reindex(times)
u, v = W.u.values, W.v.values
spd = np.hypot(u, v)
# steadiness over [t, t+6]: |vector mean| / scalar mean
uu = pd.Series(u).rolling(7).mean().shift(-6).values; vv = pd.Series(v).rolling(7).mean().shift(-6).values
ss = pd.Series(spd).rolling(7).mean().shift(-6).values
B = np.hypot(uu, vv) / np.maximum(ss, 1e-6)
mspd = ss  # mean speed over the window

# local coordinates (km)
lat0, lon0 = lat.mean(), lon.mean()
x = (lon - lon0) * 111.32 * np.cos(np.deg2rad(lat0)); y = (lat - lat0) * 110.57
# 2) Hourly change of log PM2.5. Differencing removes slow, city-wide level changes, so the
#    remaining co-movement between stations reflects short-term propagation.
dz = np.diff(np.log(pm), axis=0, prepend=np.nan)  # hourly change of log PM2.5
if DESEASON:  # remove station-specific hour-of-day x month mean of dz (diurnal-cycle confound)
    key = times.hour.values * 100 + times.month.values
    D = pd.DataFrame(dz); D["k"] = key
    dz = (D.drop(columns="k") - D.groupby("k").transform("mean").drop(columns="k", errors="ignore")).values

# 3) Lags examined (hours) and the 'steady wind' condition used to select hours.
LAGS = np.arange(-8, 9)
wind_ang = np.arctan2(vv, uu)  # direction of mean motion over window
ok_wind = (mspd >= 1.5) & (B >= 0.8)

# Pearson correlation between station a at time t and station b at time t+lag,
# using only the selected hours; returns NaN if fewer than 150 valid pairs.
def lagcorr(a, b, idx, lag):
    j = idx + lag; m = (j >= 0) & (j < len(b)); ii, jj = idx[m], j[m]
    xa, xb = a[ii], b[jj]; f = np.isfinite(xa) & np.isfinite(xb)
    if f.sum() < 150: return np.nan, f.sum()
    return np.corrcoef(xa[f], xb[f])[0, 1], f.sum()

# =====================================================================================
# 4) For every ordered pair 3-30 km apart, compare lagged correlations on hours when the wind
#    blows from i towards j (downwind set) with hours when it blows from j towards i (upwind set).
#    A = mean over lags 1-3 h of (c_down - c_up); transport predicts A > 0.
# =====================================================================================
rows = []
for i in range(N):
    for j in range(N):
        if i == j: continue
        dx, dy = x[j] - x[i], y[j] - y[i]; d = np.hypot(dx, dy)
        if d < 3 or d > 30: continue
        bear = np.arctan2(dy, dx)
        diff = np.angle(np.exp(1j * (wind_ang - bear)))
        down = np.where(ok_wind & (np.abs(diff) <= np.deg2rad(30)))[0]
        up = np.where(ok_wind & (np.abs(np.angle(np.exp(1j * (wind_ang - bear - np.pi)))) <= np.deg2rad(30)))[0]
        if len(down) < 300 or len(up) < 300: continue
        cd = np.array([lagcorr(dz[:, i], dz[:, j], down, L)[0] for L in LAGS])
        cu = np.array([lagcorr(dz[:, i], dz[:, j], up, L)[0] for L in LAGS])
        if np.isnan(cd).any() or np.isnan(cu).any(): continue
        s_down = np.nanmean(mspd[down]) * np.cos(np.nanmean(np.abs(diff[down])))
        T = d / (s_down * 3.6)  # hours
        pos = (LAGS >= 1) & (LAGS <= 3)
        rows.append(dict(i=i, j=j, d_km=d, n_down=len(down), n_up=len(up), T_h=T,
                         A=np.mean(cd[pos]) - np.mean(cu[pos]),
                         tau_star_down=int(LAGS[np.argmax(cd)]), tau_star_up=int(LAGS[np.argmax(cu)]),
                         **{f"cd_{L}": c for L, c in zip(LAGS, cd)}, **{f"cu_{L}": c for L, c in zip(LAGS, cu)}))

# 5) Summary statistics across pairs: sign test via Wilcoxon, lag profile, and whether the
#    lag of maximum correlation grows with the advective travel time (Spearman).
R = pd.DataFrame(rows); R.to_csv(os.path.join(out, "lag_pairs.csv"), index=False)
# unordered-pair average of A to avoid double counting
R["key"] = [tuple(sorted(k)) for k in zip(R.i, R.j)]
Au = R.groupby("key").A.mean()
w = stats.wilcoxon(Au.values)
# lead structure: is the downwind-minus-upwind curve asymmetric (positive lags > negative lags)?
cdm = R[[f"cd_{L}" for L in LAGS]].values - R[[f"cu_{L}" for L in LAGS]].values
mean_curve = cdm.mean(axis=0)
sp = stats.spearmanr(R.T_h, R.tau_star_down)
res = dict(n_ordered_pairs=len(R), n_unordered_pairs=int(len(Au)),
           median_A=float(np.median(Au)), mean_A=float(Au.mean()), frac_A_positive=float((Au > 0).mean()),
           wilcoxon_p=float(w.pvalue),
           mean_down_minus_up_by_lag={int(L): float(m) for L, m in zip(LAGS, mean_curve)},
           spearman_T_vs_tau_star=dict(rho=float(sp.statistic), p=float(sp.pvalue)),
           median_T_h=float(R.T_h.median()), median_tau_star_down=float(R.tau_star_down.median()),
           frac_tau_star_down_positive=float((R.tau_star_down > 0).mean()),
           frac_tau_star_up_positive=float((R.tau_star_up > 0).mean()),
           frac_hours_steady_wind=float(np.nanmean(ok_wind)))
json.dump(res, open(os.path.join(out, "lag_test.json"), "w"), indent=1)
print(json.dumps(res, indent=1))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].axhline(0, c="grey", lw=0.8); ax[0].plot(LAGS, mean_curve, marker="o", c="#1f5fa8")
ax[0].set_xlabel("lag tau (h)"); ax[0].set_ylabel("c_down - c_up"); ax[0].set_title("Downwind minus upwind lagged corr.")
ax[1].hist(Au.values, bins=30, color="#1f5fa8"); ax[1].axvline(0, c="k", lw=0.8)
ax[1].set_xlabel("asymmetry A (lags 1-3 h)"); ax[1].set_ylabel("station pairs"); ax[1].set_title("Per-pair asymmetry")
plt.tight_layout(); plt.savefig(os.path.join(out, "fig_lag_test.png"), dpi=200)


**`graph_experiment.py`** - builds the seven graphs, the shared features and the neighbour aggregates, then trains and evaluates XGBoost for every arm and horizon.

In [ ]:
%%writefile src/graph_experiment.py
"""Step 3: controlled comparison of graph constructions with a fixed learner (XGBoost, GPU on Colab).

The graph enters ONLY through neighbour-aggregate features m_j(t-l) = sum_i P_ji x_i(t-l) / sum_i P_ji
(over observed neighbours) plus the edge mass sum_i P_ji. Everything else is identical across arms.
Meteorology is used only at the forecast origin t (no future weather).

Usage: python graph_experiment.py <prep_dir> <out_dir> [test_year]
"""
import sys, os, json, time
import numpy as np, pandas as pd, xgboost as xgb
try:
    import torch; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
print("xgboost", xgb.__version__, "device", DEVICE, flush=True)

prep, out = sys.argv[1], sys.argv[2]; os.makedirs(out, exist_ok=True)
# =====================================================================================
# 0) Load the hourly panel and define the rolling-origin split for the chosen test year.
# =====================================================================================
Z = np.load(os.path.join(prep, "panel.npz"), allow_pickle=True)
X = Z["pm"].astype(np.float32); lat, lon = Z["lat"], Z["lon"]
times = pd.to_datetime(Z["times"].astype("datetime64[h]")); T, N = X.shape
HORIZONS = [1, 3, 6, 12, 24]; NLAGS = [0, 1, 2, 3, 6]
yr = times.year.values
TEST_YEAR = int(sys.argv[3]) if len(sys.argv) > 3 else 2024   # rolling origin: train < TEST-1, val = TEST-1
train_o = (yr <= TEST_YEAR - 2) & (np.arange(T) >= 24); val_o = yr == TEST_YEAR - 1; test_o = yr == TEST_YEAR

# ---------------- meteorology at origin (city mean over ERA5 grid points) ----------------
met = pd.read_parquet(os.path.join(prep, "met.parquet"))
def uv(s, d): r = np.deg2rad(d); return -s * np.sin(r), -s * np.cos(r)
met["u100"], met["v100"] = uv(met.wind_speed_100m, met.wind_direction_100m)
met["u10"], met["v10"] = uv(met.wind_speed_10m, met.wind_direction_10m)
M = met.groupby("time")[["u100", "v100", "u10", "v10", "boundary_layer_height"]].mean().reindex(times).interpolate(limit=3)
u, v = M.u100.values, M.v100.values; s100 = np.hypot(u, v); s10 = np.hypot(M.u10.values, M.v10.values)
blh = M.boundary_layer_height.values; V = np.maximum(blh, 50) * np.maximum(s10, 0.1)
lnV0 = np.nanmedian(np.log(V[train_o]))

# ---------------- geometry ----------------
lat0, lon0 = lat.mean(), lon.mean()
xk = (lon - lon0) * 111.32 * np.cos(np.deg2rad(lat0)); yk = (lat - lat0) * 110.57
DX = xk[None, :] - xk[:, None]; DY = yk[None, :] - yk[:, None]          # [i, j]: vector i -> j
D = np.hypot(DX, DY); np.fill_diagonal(D, np.inf)
EX, EY = DX / D, DY / D                                                  # unit bearing i -> j
def K(sig): k = np.exp(-(D ** 2) / (2 * sig ** 2)); return k            # diag -> 0 (D=inf)

# correlation graph (train only): top-5 correlated neighbours
C = pd.DataFrame(X[train_o]).corr().values; np.fill_diagonal(C, -np.inf)
Pcorr = np.zeros((N, N), np.float32)
for j in range(N):
    for i in np.argsort(C[:, j])[-5:]: Pcorr[j, i] = max(C[i, j], 0)

def calm(s): return 1 / (1 + np.exp(-(s - 1.0) / 0.5))

# -------------------------------------------------------------------------------------
# Neighbour aggregation for a fixed graph P (receiver j x source i): weighted mean of the
# neighbours' PM2.5 at lags 0,1,2,3,6 h, skipping missing values, plus the edge mass.
# -------------------------------------------------------------------------------------
def static_agg(P):
    """P[j, i] receiver j, source i. returns agg[T, N, len(NLAGS)], mass[N]"""
    mask = np.isfinite(X).astype(np.float32); X0 = np.nan_to_num(X)
    feats = []
    for l in NLAGS:
        Xl = np.roll(X0, l, 0); Ml = np.roll(mask, l, 0); Xl[:l] = 0; Ml[:l] = 0
        num = Xl @ P.T; den = Ml @ P.T
        feats.append(np.where(den > 1e-6, num / np.maximum(den, 1e-6), np.nan))
    return np.stack(feats, -1), np.repeat(P.sum(1)[None, :], T, 0)

# Hourly wind graph: distance kernel x positive cosine between wind and bearing x calm gate.
# With 'gate', mix it with the distance graph according to ventilation (stagnant air -> distance).
def dynamic_wind_P(sig, gate=None):
    """P_t[j, i] = K_ij * relu(cos(wind_t, bearing i->j)) * calm(s_t); optional hybrid with distance."""
    cosang = (u[:, None, None] * EX[None] + v[:, None, None] * EY[None]) / np.maximum(s100, 1e-3)[:, None, None]
    Pw = K(sig)[None] * np.maximum(cosang, 0) * calm(s100)[:, None, None]   # [t, i, j]
    Pw = np.transpose(Pw, (0, 2, 1)).astype(np.float32)                        # [t, j, i]
    if gate is not None:
        g = gate[:, None, None]
        Pw = g * Pw + (1 - g) * K(sig).T[None].astype(np.float32)
    return np.nan_to_num(Pw)

# Same aggregation as static_agg, but with a different graph at every forecast origin t.
def dynamic_agg(Pt):
    mask = np.isfinite(X).astype(np.float32); X0 = np.nan_to_num(X)
    feats = []
    for l in NLAGS:
        Xl = np.roll(X0, l, 0); Ml = np.roll(mask, l, 0); Xl[:l] = 0; Ml[:l] = 0
        num = np.einsum("tji,ti->tj", Pt, Xl); den = np.einsum("tji,ti->tj", Pt, Ml)
        feats.append(np.where(den > 1e-6, num / np.maximum(den, 1e-6), np.nan))
    return np.stack(feats, -1), Pt.sum(2)

# =====================================================================================
# 1) Features shared by every arm. Only the neighbour-aggregate block differs between arms.
# =====================================================================================
# ---------------- base (graph-free) features ----------------
def lagged(A, l): B = np.roll(A, l, 0); B[:l] = np.nan; return B
own = np.stack([lagged(X, l) for l in [0, 1, 2, 3, 4, 5, 11, 23]], -1)
roll24 = pd.DataFrame(X).rolling(24, min_periods=12).mean().values
cal_hour = times.hour.values; cal_month = times.month.values; cal_dow = times.dayofweek.values
metf = np.stack([u, v, s100, s10, np.log(np.maximum(blh, 1)), np.log(V)], -1)

# Build the design matrix for a set of forecast origins and horizon h.
# Columns: own lags | 24 h mean | calendar + station id | origin meteorology | [graph block]
def build(idx, h, gfeat=None, gmass=None):
    tt = np.repeat(idx, N); jj = np.tile(np.arange(N), len(idx))
    tgt = tt + h; okm = tgt < T
    tt, jj, tgt = tt[okm], jj[okm], tgt[okm]
    y = X[tgt, jj]
    cols = [own[tt, jj], roll24[tt, jj][:, None],
            np.stack([(cal_hour[tt] + h) % 24, cal_month[tt], cal_dow[tt], jj], -1).astype(np.float32),
            metf[tt]]
    if gfeat is not None:
        cols += [gfeat[tt, jj], gmass[tt, jj][:, None]]
    F = np.concatenate(cols, 1).astype(np.float32)
    keep = np.isfinite(y) & np.isfinite(F[:, 0])
    return F[keep], y[keep], tt[keep], jj[keep]

# 2) Fixed learner: XGBoost, absolute-error loss, GPU histogram method.
PARAMS = dict(objective="reg:absoluteerror", eta=0.08, max_depth=8, min_child_weight=50, subsample=0.8,
              colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist", device=DEVICE, max_bin=256, seed=0)
tr_idx = np.where(train_o)[0][::2]; va_idx = np.where(val_o)[0]; te_idx = np.where(test_o)[0]

# Train on the change from the current value (so every arm starts at persistence), stop early on
# the validation year, and return test predictions in original units plus the validation MAE.
def fit_eval(gfeat, gmass, h, idx_eval):
    Ftr, ytr, _, _ = build(tr_idx, h, gfeat, gmass); Fva, yva, _, _ = build(va_idx, h, gfeat, gmass)
    # target = change from the station's current value, so every arm starts from persistence
    dtr = xgb.DMatrix(Ftr, ytr - Ftr[:, 0]); dva = xgb.DMatrix(Fva, yva - Fva[:, 0])
    m = xgb.train(PARAMS, dtr, 4000, evals=[(dva, "val")], early_stopping_rounds=150, verbose_eval=False)
    Fe, ye, te, je = build(idx_eval, h, gfeat, gmass)
    rng_ = (0, m.best_iteration + 1)
    pe = Fe[:, 0] + m.predict(xgb.DMatrix(Fe), iteration_range=rng_)
    pv = Fva[:, 0] + m.predict(dva, iteration_range=rng_)
    return m, pe, ye, te, je, float(np.mean(np.abs(pv - yva)))

log = {}
t0 = time.time()
# =====================================================================================
# 3) Graph hyperparameters chosen on the validation year only (6 h horizon).
# ---- tune sigma for distance graph and gate slope on validation (h=6) ----
sig_scores = {}
for sig in [5, 10, 20]:
    gf, gm = static_agg(K(sig).T.astype(np.float32))
    sig_scores[sig] = fit_eval(gf, gm, 6, va_idx)[5]
SIG = min(sig_scores, key=sig_scores.get); log["sigma_val_mae"] = sig_scores; log["sigma"] = SIG
print("sigma val MAE", sig_scores, "->", SIG, f"{time.time()-t0:.0f}s", flush=True)

gate_scores = {}
for a in [1.0, 3.0]:
    g = 1 / (1 + np.exp(-a * (np.log(V) - lnV0)))
    gf, gm = dynamic_agg(dynamic_wind_P(SIG, gate=np.nan_to_num(g, nan=0.5)))
    gate_scores[a] = fit_eval(gf, gm, 6, va_idx)[5]
A_GATE = min(gate_scores, key=gate_scores.get); log["gate_val_mae"] = gate_scores; log["gate_a"] = A_GATE
print("gate val MAE", gate_scores, "->", A_GATE, f"{time.time()-t0:.0f}s", flush=True)

# =====================================================================================
# 4) The seven graph constructions compared in the paper (G0 = no graph).
# =====================================================================================
# ---- build arms ----
ubar, vbar = np.nanmean(u[train_o]), np.nanmean(v[train_o]); sbar = np.hypot(ubar, vbar)
Pstatic = (K(SIG) * np.maximum((ubar * EX + vbar * EY) / sbar, 0)).T.astype(np.float32)
gate = np.nan_to_num(1 / (1 + np.exp(-A_GATE * (np.log(V) - lnV0))), nan=0.5)
arms = {"G0_none": (None, None),
        "G1_uniform": static_agg(np.ones((N, N), np.float32) - np.eye(N, dtype=np.float32)),
        "G2_distance": static_agg(K(SIG).T.astype(np.float32)),
        "G3_correlation": static_agg(Pcorr),
        "G4_static_wind": static_agg(Pstatic),
        "G5_hourly_wind": dynamic_agg(dynamic_wind_P(SIG)),
        "G6_vent_gated": dynamic_agg(dynamic_wind_P(SIG, gate=gate))}
log["static_wind_dir_towards_deg"] = float(np.degrees(np.arctan2(ubar, vbar)) % 360)

# 5) Train and evaluate every arm at every horizon; baselines are added to the same table.
rows = []; preds = []
for h in HORIZONS:
    base = None
    for name, (gf, gm) in arms.items():
        m, p, y, tt, jj, vmae = fit_eval(gf, gm, h, te_idx)
        if base is None:
            base = pd.DataFrame({"t": tt, "j": jj, "h": h, "y": y})
            base["persistence"] = X[tt, jj]
            base["seasonal_naive"] = X[tt + h - 24, jj] if h <= 24 else np.nan
        base[name] = p
        rows.append(dict(h=h, arm=name, test_mae=float(np.mean(np.abs(p - y))), val_mae=vmae, n_trees=int(m.best_iteration)))
        print(h, name, round(rows[-1]["test_mae"], 3), f"{time.time()-t0:.0f}s", flush=True)
    preds.append(base)
P = pd.concat(preds, ignore_index=True)
P.to_parquet(os.path.join(out, "test_predictions.parquet"), index=False)
pd.DataFrame(rows).to_csv(os.path.join(out, "summary_mae.csv"), index=False)
# met at origin for regime analysis
pd.DataFrame({"t": np.arange(T), "time": times, "u100": u, "v100": v, "s100": s100, "s10": s10, "blh": blh, "V": V,
              "gate": gate}).to_parquet(os.path.join(out, "met_origin.parquet"), index=False)
np.savez(os.path.join(out, "geometry.npz"), EX=EX, EY=EY, D=np.where(np.isinf(D), 0, D), lnV0=lnV0)
json.dump(log, open(os.path.join(out, "tuning.json"), "w"), indent=1)
print("done", f"{time.time()-t0:.0f}s")


**`evaluate_stats.py`** - metrics, Diebold-Mariano tests with Holm correction, week-block bootstrap, regime and transport-ratio breakdowns.

In [ ]:
%%writefile src/evaluate_stats.py
"""Step 4: statistics and regime analysis on test-set (2024) predictions.

Usage: python evaluate_stats.py <predictions.parquet> <met_origin.parquet> <panel.npz> <out_dir> <tag>
Works for both the XGBoost and the GNN prediction files (columns t, j, h, y, <arms...>).
"""
import sys, os, json
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

pred_f, met_f, panel_f, out, tag = sys.argv[1:6]; os.makedirs(out, exist_ok=True)
P = pd.read_parquet(pred_f); met = pd.read_parquet(met_f).set_index("t")
Z = np.load(panel_f, allow_pickle=True); lat, lon = Z["lat"], Z["lon"]
times = pd.to_datetime(Z["times"].astype("datetime64[h]"))
arms = [c for c in P.columns if c.startswith("G")]
base_arm = "G0_none"
others = [c for c in ["persistence", "seasonal_naive"] if c in P.columns]
rng = np.random.default_rng(0)

# =====================================================================================
# Regime labels for every forecast (used for evaluation only, never as model inputs):
# season of the target hour, ventilation tercile at the origin, transport-ratio bin.
# =====================================================================================
# ---------- regimes ----------
tt = P.t.values; tgt = tt + P.h.values
ttime = times[np.minimum(tgt, len(times) - 1)]
md = ttime.month * 100 + ttime.day
season = np.select([(md >= 1016) & (md <= 1130), (md >= 1216) | (md <= 115), (ttime.month >= 4) & (ttime.month <= 6),
                    (ttime.month >= 7) & (ttime.month <= 9)], ["stubble", "peak_winter", "pre_monsoon", "monsoon"], "other")
P["season"] = season
trV = met.loc[(times.year <= 2022)[met.index.values], "V"]
q1, q2 = np.nanquantile(np.log(trV), [1 / 3, 2 / 3])
lv = np.log(met.loc[tt, "V"].values)
P["vent"] = np.select([lv < q1, lv < q2], ["low", "mid"], "high")
# transport ratio: s100 * h / distance to nearest upwind station within +-45 deg
lat0, lon0 = lat.mean(), lon.mean()
xk = (lon - lon0) * 111.32 * np.cos(np.deg2rad(lat0)); yk = (lat - lat0) * 110.57
u = met.loc[tt, "u100"].values; v = met.loc[tt, "v100"].values; s = np.hypot(u, v)
jj = P.j.values
dx = xk[jj][:, None] - xk[None, :]; dy = yk[jj][:, None] - yk[None, :]   # vector source i -> receiver j
d = np.hypot(dx, dy); d[np.arange(len(jj)), jj] = np.inf
cosang = (u[:, None] * dx + v[:, None] * dy) / (np.maximum(s, 1e-3)[:, None] * np.maximum(d, 1e-6))
dup = np.where((cosang >= np.cos(np.deg2rad(45))) & np.isfinite(d), d, np.inf).min(1)
rho = s * 3.6 * P.h.values / dup
P["rho_bin"] = np.select([~np.isfinite(dup), rho < 0.5, rho <= 2], ["no_upwind", "<0.5", "0.5-2"], ">2")
P["date"] = ttime.normalize(); P["week"] = ttime.to_period("W").astype(str)

# dm_hln: Diebold-Mariano test with Harvey-Leybourne-Newbold correction on daily loss differences
# nw_tstat: t-test with Newey-West standard errors
# block_boot_ci: 95% CI of an MAE difference, resampling whole ISO weeks
# ---------- helpers ----------
def dm_hln(d, h_lag=1):
    d = d[np.isfinite(d)]; n = len(d); dbar = d.mean()
    g = [np.mean((d[k:] - dbar) * (d[:n - k] - dbar)) for k in range(h_lag)]
    var = (g[0] + 2 * sum(g[1:])) / n
    stat = dbar / np.sqrt(var) * np.sqrt((n + 1 - 2 * h_lag + h_lag * (h_lag - 1) / n) / n)
    return stat, 2 * stats.t.sf(abs(stat), n - 1)

def nw_tstat(x, lag=7):
    x = x[np.isfinite(x)]; n = len(x); xb = x.mean(); e = x - xb
    s2 = e @ e / n + 2 * sum((1 - k / (lag + 1)) * (e[k:] @ e[:-k]) / n for k in range(1, lag + 1))
    t = xb / np.sqrt(s2 / n); return xb, t, 2 * stats.t.sf(abs(t), n - 1)

def block_boot_ci(df, a, b, B=1000):
    """95% CI of MAE(a) - MAE(b), resampling ISO weeks."""
    g = df.assign(ea=(df[a] - df.y).abs(), eb=(df[b] - df.y).abs()).groupby("week")[["ea", "eb"]].agg(["sum", "count"])
    sa, sb, n = g[("ea", "sum")].values, g[("eb", "sum")].values, g[("ea", "count")].values
    W = len(n); res = []
    for _ in range(B):
        k = rng.integers(0, W, W); res.append(sa[k].sum() / n[k].sum() - sb[k].sum() / n[k].sum())
    return np.percentile(res, [2.5, 97.5])

# Overall accuracy by horizon (MAE, RMSE, R2), each arm vs. the no-graph arm, Holm-adjusted.
# ---------- overall table + DM tests ----------
rows = []
for h, g in P.groupby("h"):
    m0 = (g[base_arm] - g.y).abs().mean()
    for a in arms + others:
        mae = (g[a] - g.y).abs().mean(); rmse = float(np.sqrt(((g[a] - g.y) ** 2).mean()))
        r2 = float(1 - ((g[a] - g.y) ** 2).sum() / ((g.y - g.y.mean()) ** 2).sum())
        row = dict(h=h, model=a, mae=mae, rmse=rmse, r2=r2, pct_vs_G0=100 * (mae - m0) / m0)
        if a != base_arm:
            daily = g.assign(e=(g[a] - g.y).abs() - (g[base_arm] - g.y).abs()).groupby("date").e.mean().values
            st, p = dm_hln(daily); lo, hi = block_boot_ci(g, a, base_arm)
            row.update(dm_stat=st, dm_p=p, ci_lo=lo, ci_hi=hi)
        rows.append(row)
R = pd.DataFrame(rows)
prim = R[R.model.isin([a for a in arms if a != base_arm])].copy()
order = prim.dm_p.sort_values().index; m = len(order); holm = pd.Series(index=order, dtype=float); run = 0
for r, ix in enumerate(order): run = max(run, min(1, (m - r) * prim.loc[ix, "dm_p"])); holm[ix] = run
R.loc[holm.index, "dm_p_holm"] = holm
R.to_csv(os.path.join(out, f"{tag}_overall.csv"), index=False)

# wind vs uniform/distance (does direction add anything beyond "use the neighbours"?)
pair_rows = []
for h, g in P.groupby("h"):
    for a, b in [("G5_hourly_wind", "G2_distance"), ("G5_hourly_wind", "G1_uniform"), ("G6_vent_gated", "G2_distance"),
                 ("G5_hourly_wind", "G4_static_wind"), ("G4_static_wind", "G2_distance")]:
        if a in g and b in g:
            daily = g.assign(e=(g[a] - g.y).abs() - (g[b] - g.y).abs()).groupby("date").e.mean().values
            st, p = dm_hln(daily); lo, hi = block_boot_ci(g, a, b)
            mb = (g[b] - g.y).abs().mean(); ma = (g[a] - g.y).abs().mean()
            pair_rows.append(dict(h=h, a=a, b=b, mae_a=ma, mae_b=mb, pct=100 * (ma - mb) / mb, dm_p=p, ci_lo=lo, ci_hi=hi))
pd.DataFrame(pair_rows).to_csv(os.path.join(out, f"{tag}_pairwise.csv"), index=False)

# ---------- regime tables ----------
reg_rows = []
for col in ["season", "vent", "rho_bin"]:
    for (h, lev), g in P.groupby(["h", col]):
        m0 = (g[base_arm] - g.y).abs().mean()
        r = dict(h=h, regime=col, level=lev, n=len(g), mae_G0=m0)
        for a in arms:
            if a == base_arm: continue
            r[f"pct_{a}"] = 100 * ((g[a] - g.y).abs().mean() - m0) / m0
        if "G5_hourly_wind" in g and "G2_distance" in g:
            r["pct_G5_vs_G2"] = 100 * ((g.G5_hourly_wind - g.y).abs().mean() / (g.G2_distance - g.y).abs().mean() - 1)
        if "persistence" in g: r["pct_persist"] = 100 * ((g.persistence - g.y).abs().mean() - m0) / m0
        reg_rows.append(r)
RG = pd.DataFrame(reg_rows); RG.to_csv(os.path.join(out, f"{tag}_regimes.csv"), index=False)

# ---------- H3: is the wind-graph gain larger at intermediate transport ratio? ----------
h3 = []
for comp in [("G5_hourly_wind", "G0_none"), ("G5_hourly_wind", "G2_distance")]:
    a, b = comp
    if a not in P or b not in P: continue
    for h, g in P.groupby("h"):
        g = g.assign(e=(g[a] - g.y).abs() - (g[b] - g.y).abs())
        mid = g[g.rho_bin == "0.5-2"].groupby("date").e.mean(); rest = g[g.rho_bin != "0.5-2"].groupby("date").e.mean()
        dd = (mid - rest).dropna().values
        if len(dd) < 20: h3.append(dict(a=a, b=b, h=h, n_days=len(dd))); continue
        xb, t, p = nw_tstat(dd)
        h3.append(dict(a=a, b=b, h=h, n_days=len(dd), mean_mid_minus_rest=xb, t=t, p=p,
                       share_mid=float((g.rho_bin == "0.5-2").mean())))
pd.DataFrame(h3).to_csv(os.path.join(out, f"{tag}_H3_conditional.csv"), index=False)

# ---------- figures ----------
fig, ax = plt.subplots(figsize=(6.2, 3.2))
HS = sorted(P.h.unique()); w = 0.8 / (len(arms) - 1)
for k, a in enumerate([x for x in arms if x != base_arm]):
    vals = [R[(R.h == h) & (R.model == a)].pct_vs_G0.values[0] for h in HS]
    ax.bar(np.arange(len(HS)) + k * w - 0.4 + w / 2, vals, w, label=a.split("_", 1)[1].replace("_", " "))
ax.axhline(0, c="k", lw=0.8); ax.set_xticks(range(len(HS))); ax.set_xticklabels([f"{h} h" for h in HS])
ax.set_ylabel("MAE change vs. no graph (%)"); ax.legend(fontsize=6.5, ncol=2, frameon=False)
plt.tight_layout(); plt.savefig(os.path.join(out, f"{tag}_fig_overall.png"), dpi=220)

if "G5_hourly_wind" in P and "G2_distance" in P:
    fig, ax = plt.subplots(figsize=(6.2, 3.0))
    sub = RG[RG.regime == "rho_bin"]
    for lev, c in zip(["<0.5", "0.5-2", ">2", "no_upwind"], ["#9ecae1", "#1f5fa8", "#08306b", "#bbbbbb"]):
        q = sub[sub.level == lev].sort_values("h")
        ax.plot(q.h, q.pct_G5_vs_G2, marker="o", c=c, label=f"rho {lev}")
    ax.axhline(0, c="k", lw=0.8); ax.set_xlabel("horizon (h)"); ax.set_ylabel("MAE: hourly wind vs distance (%)")
    ax.legend(fontsize=7, frameon=False); plt.tight_layout(); plt.savefig(os.path.join(out, f"{tag}_fig_rho.png"), dpi=220)

print(R.round(3).to_string())
print(pd.DataFrame(pair_rows).round(3).to_string())
print(pd.DataFrame(h3).round(4).to_string())


**`gnn_experiment.py`** - compact Graph WaveNet-style network trained once per graph arm and seed.

In [ ]:
%%writefile src/gnn_experiment.py
"""Step 5 (run on Google Colab, GPU): robustness check with a compact Graph WaveNet-style model.

Same data, splits, horizons and graph constructions as the XGBoost experiment; the only thing
that changes between arms is the support matrix P (receiver x source) used for message passing.

Usage (Colab): python gnn_experiment.py <panel.npz> <met_origin.parquet> <tuning.json> <out_dir>
"""
import os, sys, json, time, math
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

dev = "cuda" if torch.cuda.is_available() else "cpu"; print("device", dev)
PANEL, METF, TUNF, OUT = sys.argv[1:5]; os.makedirs(OUT, exist_ok=True)
Z = np.load(PANEL, allow_pickle=True)
X = Z["pm"].astype(np.float32); lat, lon = Z["lat"], Z["lon"]
times = pd.to_datetime(Z["times"].astype("datetime64[h]")); T, N = X.shape
M = pd.read_parquet(METF)
u, v, s100, s10, blh, Vv, gate = [M[c].values.astype(np.float32) for c in ["u100", "v100", "s100", "s10", "blh", "V", "gate"]]
HOR = [1, 3, 6, 12, 24]; L = 24; yr = times.year.values
SIG = float(json.load(open(TUNF))["sigma"])

# =====================================================================================
# 1) Graphs - identical definitions to graph_experiment.py; only the support P changes per arm.
# =====================================================================================
lat0, lon0 = lat.mean(), lon.mean()
xk = (lon - lon0) * 111.32 * np.cos(np.deg2rad(lat0)); yk = (lat - lat0) * 110.57
DX = xk[None, :] - xk[:, None]; DY = yk[None, :] - yk[:, None]; D = np.hypot(DX, DY); np.fill_diagonal(D, np.inf)
EX, EY = DX / D, DY / D
Kd = np.exp(-(D ** 2) / (2 * SIG ** 2))                     # [i, j], zero diagonal
dK = Kd.T.sum(1, keepdims=True)                              # fixed denominator per receiver
P_dist = (Kd.T / dK).astype(np.float32)                      # [j, i]
TEST_YEAR = int(os.environ.get("TEST_YEAR", "2024"))   # train <= TEST-2, validate TEST-1, test TEST
train_o = (yr <= TEST_YEAR - 2)
ub, vb = np.nanmean(u[train_o]), np.nanmean(v[train_o]); sb = np.hypot(ub, vb)
P_static = ((Kd * np.maximum((ub * EX + vb * EY) / sb, 0)).T / dK).astype(np.float32)
calm = lambda s: 1 / (1 + np.exp(-(s - 1.0) / 0.5))

tEX, tEY = torch.tensor(np.nan_to_num(EX), device=dev, dtype=torch.float32), torch.tensor(np.nan_to_num(EY), device=dev, dtype=torch.float32)
tKd = torch.tensor(np.nan_to_num(Kd), device=dev, dtype=torch.float32); tdK = torch.tensor(dK, device=dev, dtype=torch.float32)
tu, tv = torch.tensor(np.nan_to_num(u), device=dev), torch.tensor(np.nan_to_num(v), device=dev)
ts = torch.tensor(np.nan_to_num(s100), device=dev); tg = torch.tensor(np.nan_to_num(gate, nan=0.5), device=dev)
tPdist = torch.tensor(P_dist, device=dev)

def P_wind(ii, gated=False):
    """ii: LongTensor of origin indices. returns [B, N(receiver), N(source)] on GPU."""
    uu, vv, ss = tu[ii], tv[ii], ts[ii].clamp_min(1e-3)
    cos = (uu[:, None, None] * tEX[None] + vv[:, None, None] * tEY[None]) / ss[:, None, None]
    calm_ = torch.sigmoid((ts[ii] - 1.0) / 0.5)
    Pw = (tKd[None] * cos.clamp_min(0) * calm_[:, None, None]).transpose(1, 2) / tdK[None]
    if gated:
        g = tg[ii][:, None, None]; Pw = g * Pw + (1 - g) * tPdist[None]
    return torch.nan_to_num(Pw)

# 2) Normalise with training statistics only; forward-fill inputs and keep a missing-value mask.
# ---- data tensors ----
mu = np.nanmean(X[train_o]); sd = np.nanstd(X[train_o])
Xn = (X - mu) / sd; mask = np.isfinite(Xn); Xf = pd.DataFrame(Xn).ffill().fillna(0).values.astype(np.float32)
metf = np.stack([u, v, s100, s10, np.log(np.maximum(blh, 1)), np.log(np.maximum(Vv, 1))], -1)
metf = np.nan_to_num((metf - np.nanmean(metf[train_o], 0)) / np.nanstd(metf[train_o], 0)).astype(np.float32)
Xt = torch.tensor(Xf, device=dev); Mt = torch.tensor(mask.astype(np.float32), device=dev)
Yraw = torch.tensor(np.nan_to_num(X, nan=-1.0), device=dev); METt = torch.tensor(metf, device=dev)
hourt = torch.tensor(times.hour.values, device=dev, dtype=torch.long)

def origins(sel): return np.where(sel & (np.arange(T) >= L) & (np.arange(T) < T - max(HOR)))[0]
tr, va, te = origins(yr <= TEST_YEAR - 2), origins(yr == TEST_YEAR - 1), origins(yr == TEST_YEAR)

# =====================================================================================
# 3) Compact Graph WaveNet-style model:
#    dilated gated temporal convolutions -> origin meteorology + hour -> two forward-only
#    diffusion layers Z' = Z W0 + P Z W1 + P^2 Z W2 -> per-station head for all horizons.
# =====================================================================================
class Net(nn.Module):
    def __init__(s, n, hid=48, adaptive=False):
        super().__init__()
        s.inp = nn.Conv1d(2, hid, 1)
        s.tcn = nn.ModuleList([nn.Conv1d(hid, 2 * hid, 2, dilation=d) for d in [1, 2, 4, 8]])
        s.emb = nn.Embedding(n, 8); s.metp = nn.Linear(6 + 24, hid)
        s.g0 = nn.ModuleList([nn.Linear(hid, hid) for _ in range(2)])
        s.g1 = nn.ModuleList([nn.Linear(hid, hid) for _ in range(2)])
        s.g2 = nn.ModuleList([nn.Linear(hid, hid) for _ in range(2)])
        s.out = nn.Sequential(nn.Linear(hid + 8, 64), nn.ReLU(), nn.Linear(64, len(HOR)))
        s.adaptive = adaptive
        if adaptive: s.E1 = nn.Parameter(torch.randn(n, 10) * 0.1); s.E2 = nn.Parameter(torch.randn(n, 10) * 0.1)
    def forward(s, x, m, met, hr, P):
        B, Lx, n = x.shape
        h = s.inp(torch.stack([x, m], 2).permute(0, 3, 2, 1).reshape(B * n, 2, Lx))
        for c in s.tcn:
            z = c(F.pad(h, (c.dilation[0], 0))); a, b = z.chunk(2, 1); h = h + torch.tanh(a) * torch.sigmoid(b)
        h = h[:, :, -1].reshape(B, n, -1)
        h = h + s.metp(torch.cat([met, F.one_hot(hr, 24).float()], -1))[:, None, :]
        if s.adaptive: P = torch.softmax(F.relu(s.E1 @ s.E2.T), 1)[None].expand(B, n, n)
        for k in range(2):
            p1 = torch.bmm(P, h); p2 = torch.bmm(P, p1)
            h = F.relu(s.g0[k](h) + s.g1[k](p1) + s.g2[k](p2)) + h
        e = s.emb.weight[None].expand(B, n, 8)
        return s.out(torch.cat([h, e], -1))  # [B, n, H] in normalised units

# Assemble one mini-batch: 24 h input windows, mask, origin meteorology, graph P and targets.
def batch(idx, arm):
    ii = torch.tensor(idx, device=dev)
    win = ii[:, None] + torch.arange(-L + 1, 1, device=dev)[None]
    x, m = Xt[win], Mt[win]; met = METt[ii]; hr = hourt[ii]
    if arm == "G0_none" or arm == "G7_adaptive": P = torch.zeros(len(idx), N, N, device=dev)
    elif arm == "G2_distance": P = torch.tensor(P_dist, device=dev)[None].expand(len(idx), N, N)
    elif arm == "G4_static_wind": P = torch.tensor(P_static, device=dev)[None].expand(len(idx), N, N)
    elif arm == "G5_hourly_wind": P = P_wind(ii)
    elif arm == "G6_vent_gated": P = P_wind(ii, gated=True)
    y = torch.stack([Yraw[ii + h] for h in HOR], -1)  # [B, n, H]
    return x, m, met, hr, P, y

# Predict in original units for a set of origins (no gradient).
def evaluate(model, idx, arm):
    model.eval(); out = []
    with torch.no_grad():
        for b in range(0, len(idx), 512):
            x, m, met, hr, P, y = batch(idx[b:b + 512], arm)
            out.append((model(x, m, met, hr, P) * sd + mu).cpu().numpy())
    return np.concatenate(out)

# =====================================================================================
# 4) Train every graph arm with several seeds; early stopping on the validation year.
# =====================================================================================
ARMS = ["G0_none", "G2_distance", "G4_static_wind", "G5_hourly_wind", "G6_vent_gated", "G7_adaptive"]
SEEDS = [int(x) for x in os.environ.get("SEEDS", "0,1,2").split(",")]; rows = []; allpred = {}
Yte = np.stack([X[te + h] for h in HOR], -1)  # [n_te, N, H]
for arm in ARMS:
    preds = []
    for seed in SEEDS:
        torch.manual_seed(seed); np.random.seed(seed); t0 = time.time()
        model = Net(N, adaptive=(arm == "G7_adaptive")).to(dev)
        opt = torch.optim.AdamW(model.parameters(), 1e-3, weight_decay=1e-4)
        best, bad, state = 1e9, 0, None
        for ep in range(40):
            model.train(); perm = np.random.permutation(tr)
            for b in range(0, len(perm), 256):
                x, m, met, hr, P, y = batch(perm[b:b + 256], arm)
                pr = model(x, m, met, hr, P) * sd + mu; ok = y >= 0
                loss = (pr - y).abs()[ok].mean() / 50
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5); opt.step()
            pv = evaluate(model, va, arm); Yv = np.stack([X[va + h] for h in HOR], -1)
            vm = np.nanmean(np.abs(pv - Yv))
            if vm < best - 1e-3: best, bad, state = vm, 0, {k: x.clone() for k, x in model.state_dict().items()}
            else: bad += 1
            if bad >= 5: break
        model.load_state_dict(state); p = evaluate(model, te, arm); preds.append(p)
        for hi, h in enumerate(HOR):
            rows.append(dict(arm=arm, seed=seed, h=h, val_mae=best, test_mae=float(np.nanmean(np.abs(p[..., hi] - Yte[..., hi])))))
        print(arm, seed, "val", round(best, 2), "epochs", ep + 1, f"{time.time()-t0:.0f}s", flush=True)
    allpred[arm] = np.mean(preds, 0)
pd.DataFrame(rows).to_csv(os.path.join(OUT, "gnn_summary.csv"), index=False)
recs = []
for hi, h in enumerate(HOR):
    tt = np.repeat(te, N); jj = np.tile(np.arange(N), len(te)); y = Yte[..., hi].ravel()
    d = pd.DataFrame({"t": tt, "j": jj, "h": h, "y": y})
    for arm in ARMS: d[arm] = allpred[arm][..., hi].ravel()
    recs.append(d[np.isfinite(y)])
pd.concat(recs).to_parquet(os.path.join(OUT, "gnn_test_predictions.parquet"), index=False)
print(pd.DataFrame(rows).groupby(["arm", "h"]).test_mae.agg(["mean", "std"]).round(2))


**`resources.py`** - training curves, run time, peak GPU memory and inference latency.

In [ ]:
%%writefile src/resources.py
"""Step 6: training dynamics and resource consumption (for the paper's required graphs).

Reuses the exact definitions of graph_experiment.py and gnn_experiment.py, then for one test year (default 2024):
  * XGBoost (h = 6 h): validation-MAE learning curves, graph-construction time, fit time, inference time.
  * GNN (1 seed): per-epoch training loss and validation MAE, time per epoch, peak GPU memory,
    parameter count and inference latency, for every graph arm.
Usage: python resources.py <prep_dir> <xgb_out_dir_for_test_year> <out_dir> [test_year=2024]
"""
import sys, os, json, time
prep, xgbdir, out = sys.argv[1], sys.argv[2], sys.argv[3]; os.makedirs(out, exist_ok=True)
TY = sys.argv[4] if len(sys.argv) > 4 else "2024"; os.environ["TEST_YEAR"] = TY
HERE = os.path.dirname(os.path.abspath(__file__))
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

# Re-use the exact definitions from graph_experiment.py (executed up to the training loop).
# ======================= XGBoost =======================
src = open(os.path.join(HERE, "graph_experiment.py"), encoding="utf-8").read().split("log = {}")[0]
sys.argv = ["03", prep, os.path.join(out, "_tmp"), TY]
g = {"__name__": "xgb_defs", "__file__": os.path.join(HERE, "graph_experiment.py")}
exec(src, g)
xgb = g["xgb"]; tune = json.load(open(os.path.join(xgbdir, "tuning.json")))
SIG, A = tune["sigma"], tune["gate_a"]
V, lnV0, u, v, train_o = g["V"], g["lnV0"], g["u"], g["v"], g["train_o"]
gate = np.nan_to_num(1 / (1 + np.exp(-A * (np.log(V) - lnV0))), nan=0.5)
ubar, vbar = np.nanmean(u[train_o]), np.nanmean(v[train_o]); sbar = np.hypot(ubar, vbar)
K, EX, EY, N = g["K"], g["EX"], g["EY"], g["N"]
builders = {
    "G0_none": lambda: (None, None),
    "G1_uniform": lambda: g["static_agg"](np.ones((N, N), np.float32) - np.eye(N, dtype=np.float32)),
    "G2_distance": lambda: g["static_agg"](K(SIG).T.astype(np.float32)),
    "G3_correlation": lambda: g["static_agg"](g["Pcorr"]),
    "G4_static_wind": lambda: g["static_agg"]((K(SIG) * np.maximum((ubar * EX + vbar * EY) / sbar, 0)).T.astype(np.float32)),
    "G5_hourly_wind": lambda: g["dynamic_agg"](g["dynamic_wind_P"](SIG)),
    "G6_vent_gated": lambda: g["dynamic_agg"](g["dynamic_wind_P"](SIG, gate=gate)),
}
H = 6; xres, curves = [], {}
for name, fn in builders.items():
    t0 = time.time(); gf, gm = fn(); t_graph = time.time() - t0
    Ftr, ytr, _, _ = g["build"](g["tr_idx"], H, gf, gm); Fva, yva, _, _ = g["build"](g["va_idx"], H, gf, gm)
    Fte, yte, _, _ = g["build"](g["te_idx"], H, gf, gm)
    dtr = xgb.DMatrix(Ftr, ytr - Ftr[:, 0]); dva = xgb.DMatrix(Fva, yva - Fva[:, 0])
    ev = {}; t0 = time.time()
    m = xgb.train(g["PARAMS"], dtr, 4000, evals=[(dtr, "train"), (dva, "val")], early_stopping_rounds=150,
                  evals_result=ev, verbose_eval=False)
    t_fit = time.time() - t0
    dte = xgb.DMatrix(Fte); t0 = time.time()
    p = Fte[:, 0] + m.predict(dte, iteration_range=(0, m.best_iteration + 1)); t_inf = time.time() - t0
    curves[name] = {"train": ev["train"]["mae"], "val": ev["val"]["mae"]}
    xres.append(dict(arm=name, n_features=Ftr.shape[1], graph_build_s=t_graph, fit_s=t_fit, best_iter=int(m.best_iteration),
                     infer_s=t_inf, infer_us_per_sample=1e6 * t_inf / len(yte), test_mae=float(np.mean(np.abs(p - yte)))))
    print("xgb", xres[-1], flush=True)
pd.DataFrame(xres).to_csv(os.path.join(out, "xgb_resources_h6.csv"), index=False)
json.dump(curves, open(os.path.join(out, "xgb_curves_h6.json"), "w"))
del g

# Re-use the exact definitions from gnn_experiment.py (executed up to the training loop).
# ======================= GNN =======================
import torch, torch.nn as nn
src = open(os.path.join(HERE, "gnn_experiment.py"), encoding="utf-8").read().split("ARMS = [")[0]
sys.argv = ["05", os.path.join(prep, "panel.npz"), os.path.join(xgbdir, "met_origin.parquet"), os.path.join(xgbdir, "tuning.json"), os.path.join(out, "_tmp")]
G = {"__name__": "gnn_defs", "__file__": os.path.join(HERE, "gnn_experiment.py")}
exec(src, G)
dev, Net, batch, evaluate, X, HOR = G["dev"], G["Net"], G["batch"], G["evaluate"], G["X"], G["HOR"]
tr, va, te, mu, sd, Nn = G["tr"], G["va"], G["te"], G["mu"], G["sd"], G["N"]
gres, hist = [], {}
Yv = np.stack([X[va + h] for h in HOR], -1)
for arm in ["G0_none", "G2_distance", "G4_static_wind", "G5_hourly_wind", "G6_vent_gated", "G7_adaptive"]:
    torch.manual_seed(0); np.random.seed(0)
    if dev == "cuda": torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    model = Net(Nn, adaptive=(arm == "G7_adaptive")).to(dev)
    nparam = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), 1e-3, weight_decay=1e-4)
    h_tr, h_va, h_t = [], [], []; best, bad = 1e9, 0
    for ep in range(40):
        model.train(); perm = np.random.permutation(tr); t0 = time.time(); tot, nb = 0.0, 0
        for b in range(0, len(perm), 256):
            x, m_, met, hr, P, y = batch(perm[b:b + 256], arm)
            pr = model(x, m_, met, hr, P) * sd + mu; ok = y >= 0
            loss = (pr - y).abs()[ok].mean() / 50
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5); opt.step()
            tot += float(loss) * 50; nb += 1
        if dev == "cuda": torch.cuda.synchronize()
        h_t.append(time.time() - t0); h_tr.append(tot / nb)
        vm = float(np.nanmean(np.abs(evaluate(model, va, arm) - Yv))); h_va.append(vm)
        if vm < best - 1e-3: best, bad = vm, 0
        else: bad += 1
        if bad >= 5: break
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if dev == "cuda" else float("nan")
    xb = batch(te[:512], arm)
    with torch.no_grad():
        for _ in range(3): model(*xb[:5])
        if dev == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(20): model(*xb[:5])
        if dev == "cuda": torch.cuda.synchronize()
    lat = (time.time() - t0) / 20
    hist[arm] = {"train_mae": h_tr, "val_mae": h_va, "epoch_s": h_t}
    gres.append(dict(arm=arm, params=nparam, epochs=len(h_tr), mean_epoch_s=float(np.mean(h_t)), peak_gpu_mb=peak,
                     infer_ms_per_batch512=1000 * lat, infer_us_per_station_forecast=1e6 * lat / (512 * Nn), best_val_mae=best))
    print("gnn", gres[-1], flush=True)
pd.DataFrame(gres).to_csv(os.path.join(out, "gnn_resources.csv"), index=False)
json.dump(hist, open(os.path.join(out, "gnn_curves.json"), "w"))

# ======================= figures =======================
lab = lambda a: a.split("_", 1)[1].replace("_", " ")
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.8))
for a, c in hist.items(): ax[0].plot(range(1, len(c["val_mae"]) + 1), c["val_mae"], label=lab(a))
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("validation MAE (ug/m3)"); ax[0].set_title("GNN"); ax[0].legend(fontsize=6, frameon=False)
for a, c in curves.items(): ax[1].plot(c["val"], label=lab(a), lw=1)
ax[1].set_xlabel("boosting round"); ax[1].set_ylabel("validation MAE of change"); ax[1].set_title("XGBoost, h = 6 h")
plt.tight_layout(); plt.savefig(os.path.join(out, "fig_training_curves.png"), dpi=220)

X_ = pd.DataFrame(xres); G_ = pd.DataFrame(gres)
fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.6))
ax[0].barh([lab(a) for a in X_.arm], X_.graph_build_s + X_.fit_s); ax[0].set_xlabel("XGBoost build + fit (s)")
ax[1].barh([lab(a) for a in G_.arm], G_.mean_epoch_s); ax[1].set_xlabel("GNN time / epoch (s)")
ax[2].barh([lab(a) for a in G_.arm], G_.infer_ms_per_batch512); ax[2].set_xlabel("GNN inference / 512 origins (ms)")
for a_ in ax[1:]: a_.set_yticklabels([])
plt.tight_layout(); plt.savefig(os.path.join(out, "fig_resources.png"), dpi=220)
print("done")


## 4. Download the data
Pulls only the Delhi rows (about 75 MB in total) from the public archive of the Central Pollution Control Board portal, one file per year.

**Output:** `data/delhi_2019.parquet` ... `data/delhi_2025.parquet`

In [ ]:
# Years 2019 .. END_YEAR; files already present are skipped
yrs = ' '.join(str(y) for y in range(2019, END_YEAR + 1))
!python src/download_data.py data {yrs}
!ls -la data

## 5. Build the hourly panel and meteorology
* Treats archive timestamps as local time (IST); aggregates to hourly values.
* Quality control: removes values below 2 or at least 999 ug/m3, values above PM10, and stuck sensors (6+ identical hours).
* Keeps stations at least 75% complete in every year 2019-2022 and at least 50% complete in every later year.
* Downloads ERA5 10 m / 100 m wind and boundary-layer height for the grid points covering the stations.

**Output:** `out/prep/panel.npz`, `out/prep/met.parquet`, `out/prep/stations.csv`

In [ ]:
if not os.path.exists('out/prep/panel.npz'):
    !python src/prepare_data.py data data/station_coords.csv out/prep {END_YEAR}
else:
    print('prep exists - skipping')

import pandas as pd
st = pd.read_csv('out/prep/stations.csv')
print(len(st), 'stations kept')
st.head()

## 6. Model-free transport test
Before training any model we check whether pollution actually travels between stations. For each pair of stations we correlate hourly PM2.5 changes at one station with changes at the other a few hours later, separately for hours when the wind blows from the first towards the second (downwind) and the reverse (upwind). If changes travel with the wind, the downwind correlation is higher at small positive lags. The test is run on raw changes and after removing each station's daily cycle.

**Output:** `out/lag/`, `out/lag_deseason/` (`lag_test.json`, `lag_pairs.csv`, `fig_lag_test.png`)

In [ ]:
if not os.path.exists('out/lag_deseason/lag_test.json'):
    !python src/lag_test.py out/prep out/lag                       # raw hourly changes
    !python src/lag_test.py out/prep out/lag_deseason deseason     # daily cycle removed

from IPython.display import Image, display
L = json.load(open('out/lag_deseason/lag_test.json'))
print({k: L[k] for k in ['n_unordered_pairs', 'median_A', 'frac_A_positive', 'wilcoxon_p',
                         'spearman_T_vs_tau_star', 'median_T_h']})
display(Image('out/lag_deseason/fig_lag_test.png'))

## 7. Main experiment: which graph helps?
For each test year we train one XGBoost model per graph construction and horizon (1, 3, 6, 12, 24 h). The learner and all other inputs are identical; only the neighbour features produced by the graph change:

| Arm | Graph |
|---|---|
| G0 | no graph (station's own history only) |
| G1 | uniform average of all other stations |
| G2 | distance kernel (width tuned on validation data) |
| G3 | correlation graph (top-5 neighbours, training data) |
| G4 | static wind graph (training-period mean wind) |
| G5 | hourly wind graph (wind at the forecast origin, calm-wind gate) |
| G6 | ventilation-gated hybrid of G5 and G2 |

Meteorology is used only at the forecast origin, so no future weather leaks into the forecasts. Each model predicts the change from the station's current value, so every arm starts from persistence. Statistics include MAE, RMSE, R2, Diebold-Mariano tests with Holm correction, bootstrap intervals and breakdowns by season, ventilation and transport ratio.

**Output per year:** `out/xgb_test{Y}/` (predictions, tuning) and `out/stats_test{Y}/` (tables, figures)

In [ ]:
# ---- test year 2025: train <= 2023, validate 2024, test 2025 -------------------------------
Y = 2025
if Y in YEARS:
    if not os.path.exists(f'out/xgb_test{Y}/summary_mae.csv'):
        !python src/graph_experiment.py out/prep out/xgb_test{Y} {Y}
    if not os.path.exists(f'out/stats_test{Y}/xgb{Y}_overall.csv'):
        !python src/evaluate_stats.py out/xgb_test{Y}/test_predictions.parquet out/xgb_test{Y}/met_origin.parquet out/prep/panel.npz out/stats_test{Y} xgb{Y} > out/stats_test{Y}.log

    # ---- show the results for this year ----
    O = pd.read_csv(f'out/stats_test{Y}/xgb{Y}_overall.csv')
    print(f'===== test year {Y} =====')
    for metric in ['mae', 'rmse', 'r2', 'pct_vs_G0']:
        print('--', metric)
        print(O.pivot(index='model', columns='h', values=metric).round(3))
else:
    print('test year', Y, 'skipped (not in YEARS)')

In [ ]:
# ---- test year 2024: train <= 2022, validate 2023, test 2024 -------------------------------
Y = 2024
if Y in YEARS:
    if not os.path.exists(f'out/xgb_test{Y}/summary_mae.csv'):
        !python src/graph_experiment.py out/prep out/xgb_test{Y} {Y}
    if not os.path.exists(f'out/stats_test{Y}/xgb{Y}_overall.csv'):
        !python src/evaluate_stats.py out/xgb_test{Y}/test_predictions.parquet out/xgb_test{Y}/met_origin.parquet out/prep/panel.npz out/stats_test{Y} xgb{Y} > out/stats_test{Y}.log

    # ---- show the results for this year ----
    O = pd.read_csv(f'out/stats_test{Y}/xgb{Y}_overall.csv')
    print(f'===== test year {Y} =====')
    for metric in ['mae', 'rmse', 'r2', 'pct_vs_G0']:
        print('--', metric)
        print(O.pivot(index='model', columns='h', values=metric).round(3))
else:
    print('test year', Y, 'skipped (not in YEARS)')

In [ ]:
# ---- test year 2023: train <= 2021, validate 2022, test 2023 -------------------------------
Y = 2023
if Y in YEARS:
    if not os.path.exists(f'out/xgb_test{Y}/summary_mae.csv'):
        !python src/graph_experiment.py out/prep out/xgb_test{Y} {Y}
    if not os.path.exists(f'out/stats_test{Y}/xgb{Y}_overall.csv'):
        !python src/evaluate_stats.py out/xgb_test{Y}/test_predictions.parquet out/xgb_test{Y}/met_origin.parquet out/prep/panel.npz out/stats_test{Y} xgb{Y} > out/stats_test{Y}.log

    # ---- show the results for this year ----
    O = pd.read_csv(f'out/stats_test{Y}/xgb{Y}_overall.csv')
    print(f'===== test year {Y} =====')
    for metric in ['mae', 'rmse', 'r2', 'pct_vs_G0']:
        print('--', metric)
        print(O.pivot(index='model', columns='h', values=metric).round(3))
else:
    print('test year', Y, 'skipped (not in YEARS)')

## 8. Robustness check with a graph neural network
Repeats the comparison with a compact Graph WaveNet-style network, so the conclusion does not depend on one learning algorithm. Only the support matrix P changes between arms; an adaptive (learned) graph is added as a sixth arm. Edges are forward-only, so information cannot flow from downwind to upwind.

**Output:** `out/gnn_test{GNN_YEAR}/` and `out/stats_gnn{GNN_YEAR}/`

In [ ]:
if not os.path.exists(f'out/gnn_test{GNN_YEAR}/gnn_summary.csv'):
    !TEST_YEAR={GNN_YEAR} SEEDS={GNN_SEEDS} python src/gnn_experiment.py out/prep/panel.npz out/xgb_test{GNN_YEAR}/met_origin.parquet out/xgb_test{GNN_YEAR}/tuning.json out/gnn_test{GNN_YEAR}
if not os.path.exists(f'out/stats_gnn{GNN_YEAR}/gnn{GNN_YEAR}_overall.csv'):
    !python src/evaluate_stats.py out/gnn_test{GNN_YEAR}/gnn_test_predictions.parquet out/xgb_test{GNN_YEAR}/met_origin.parquet out/prep/panel.npz out/stats_gnn{GNN_YEAR} gnn{GNN_YEAR} > out/stats_gnn{GNN_YEAR}.log

# mean and standard deviation of test MAE over seeds, per graph arm and horizon
Gs = pd.read_csv(f'out/gnn_test{GNN_YEAR}/gnn_summary.csv')
print(Gs.groupby(['arm', 'h']).test_mae.agg(['mean', 'std']).unstack('h').round(2))

## 9. Training dynamics and resource consumption
Retrains one model per graph arm while recording validation error per epoch / boosting round, time per epoch, peak GPU memory, parameter count and inference latency.

**Output:** `out/resources/` (`xgb_resources_h6.csv`, `gnn_resources.csv`, curves, `fig_training_curves.png`, `fig_resources.png`)

In [ ]:
if not os.path.exists('out/resources/gnn_resources.csv'):
    !TEST_YEAR={RESOURCE_YEAR} python src/resources.py out/prep out/xgb_test{RESOURCE_YEAR} out/resources {RESOURCE_YEAR}

print(pd.read_csv('out/resources/xgb_resources_h6.csv').round(3).to_string())
print(pd.read_csv('out/resources/gnn_resources.csv').round(3).to_string())
display(Image('out/resources/fig_training_curves.png'))
display(Image('out/resources/fig_resources.png'))

## 10. Package and download
`final_results.zip` contains every table, figure, log, tuning file and prediction file (the large intermediate panel files are left out because they can be rebuilt in Step 5).

In [ ]:
!rm -rf out/resources/_tmp
!rm -f final_results.zip && cd out && zip -qr ../final_results.zip . -x 'prep/panel.npz' 'prep/met.parquet'
!ls -la final_results.zip

from google.colab import files
files.download('final_results.zip')

## 11. Summary of outputs and troubleshooting

### Key output files
| Path | Contents |
|---|---|
| `out/env_info.json` | Hardware and software versions |
| `out/prep/stations.csv` | Stations used and their completeness |
| `out/lag_deseason/lag_test.json` | Model-free transport test results |
| `out/stats_test{Y}/xgb{Y}_overall.csv` | MAE, RMSE, R2 and significance tests per arm and horizon |
| `out/stats_test{Y}/xgb{Y}_pairwise.csv` | Ablation: wind vs. distance, static vs. hourly, gate |
| `out/stats_test{Y}/xgb{Y}_regimes.csv` | Breakdown by season, ventilation and transport ratio |
| `out/stats_gnn{GNN_YEAR}/` | Same tables for the graph neural network |
| `out/resources/` | Training curves and resource measurements |

### Troubleshooting
* **"GPU available: False"** - Runtime > Change runtime type > T4 GPU, then Run all again.
* **Colab disconnected** - reconnect and Run all; completed steps are skipped (outputs are in Drive).
* **GPU usage limit reached** - wait or use another account; to shorten the run set `YEARS = [2025]` and `GNN_SEEDS = "0"`.
* **Download error in Step 4** - re-run the cell; finished years are kept.
* **Open-Meteo error in Step 5** - the script retries automatically; if it still fails, re-run the cell a few minutes later.